In [1]:
import os

# Move notebook working directory to project root
os.chdir("/home/sagemaker-user/Linear-Regression-Sagemaker-Deployment")

print("Now working directory is:")
print(os.getcwd())


Now working directory is:
/home/sagemaker-user/Linear-Regression-Sagemaker-Deployment


In [2]:
import sagemaker

session = sagemaker.Session()
bucket = session.default_bucket()

# Upload file into a 'train/' prefix
session.upload_data(
    path="data/airbnb.csv",
    bucket=bucket,
    key_prefix="airbnb/data/train"
)

print(f"s3://{bucket}/airbnb/data/train/airbnb.csv")



sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
s3://sagemaker-us-east-1-149536488902/airbnb/data/train/airbnb.csv


In [ ]:
from sagemaker.sklearn.estimator import SKLearn
from sagemaker import get_execution_role
import sagemaker

session = sagemaker.Session()
role = get_execution_role()
bucket = session.default_bucket()

estimator = SKLearn(
    entry_point="train.py",
    source_dir="src",
    role=role,
    instance_type="ml.m5.large",
    framework_version="1.2-1",
    py_version="py3",
    output_path=f"s3://{bucket}/airbnb/models"
)

estimator.fit(
    inputs={
        "train": f"s3://{bucket}/airbnb/data/train"
    }
)


INFO:sagemaker:Creating training-job with name: sagemaker-scikit-learn-2026-01-22-13-15-59-249


2026-01-22 13:15:59 Starting - Starting the training job...
2026-01-22 13:16:23 Starting - Preparing the instances for training...
2026-01-22 13:16:45 Downloading - Downloading input data...
2026-01-22 13:17:15 Downloading - Downloading the training image......
2026-01-22 13:18:26 Training - Training image download completed. Training in progress.
2026-01-22 13:18:26 Uploading - Uploading generated training model/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-01-22 13:18:21,347 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-01-22 13:18:21,351 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-01-22 13:18:21,354 sa

In [8]:
from sagemaker.sklearn.model import SKLearnModel

# Create SageMaker model object using the trained artifact
model = SKLearnModel(
    model_data=estimator.model_data,   # S3 path from training job
    role=role,
    entry_point="inference.py",        # Inference script
    source_dir="src",                  # Folder containing inference.py
    framework_version="1.2-1",
    py_version="py3"
)

# Deploy model as a real-time endpoint
predictor = model.deploy(
    instance_type="ml.t2.medium",      # Free-tier safe
    initial_instance_count=1,
    endpoint_name="airbnb-linear-regression-prod-v3"
)


INFO:sagemaker:Creating model with name: sagemaker-scikit-learn-2026-01-22-16-56-03-762
INFO:sagemaker:Creating endpoint-config with name airbnb-linear-regression-prod-v3
INFO:sagemaker:Creating endpoint with name airbnb-linear-regression-prod-v3


--------------!

In [11]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

# Attach serializer/deserializer to predictor
predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

# Must EXACTLY match training feature order
sample_input = [[
    2,      # accommodates
    1.0,    # bathrooms
    1,      # bedrooms
    1,      # beds
    95.0    # review_scores_rating
]]

prediction = predictor.predict(sample_input)

print("Prediction:", prediction)


Prediction: [4.578384849632081]


In [12]:
print("predictor" in globals())


True
